# 1. — Manual GBD file download according to OECD country criterion

## Objective

This notebook documents:

1. The definition of the analytical cohort: OECD member countries.
2. The creation of the OECD member dataset.
3. The manual download procedure from the Global Burden of Disease (GBD) portal.
4. Structural validation checks of the datasets.
5. Documentation of the parameters used in the GBD download.

## Methodological decision

The primary analytical cohort is restricted to OECD member countries.

Rationale:

- Higher institutional and health system comparability.
- Reduced structural heterogeneity relative to global or SDI-based groupings.
- Clear and externally defined membership criteria.

The SDI-based approach is retained as a separate sensitivity analysis outside the main repository.

## OECD member countries source

The official list of OECD member countries is published by the OECD:

https://www.oecd.org/about/members-and-partners/

Several programmatic extraction approaches were evaluated (web scraping, OECD SDMX API, and knowledge graph queries). However:

- OECD web pages are protected by Cloudflare, preventing automated scraping.
- The OECD SDMX API provides country code lists but **does not encode OECD membership**.
- External knowledge graphs (e.g., Wikidata, DBpedia) require prior knowledge of specific data model properties.

Given that the OECD membership list is **short (38 countries), stable, and publicly accessible**, the most robust and transparent approach is to:

1. Verify the list directly from the OECD website.
2. Store it as a versioned dataset within the repository.

The canonical list used in this project is therefore stored as:

`data/3_processed/oecd_members.csv`

## 1.1 Environment and project setup

This section prepares the execution environment for the notebook.

It includes:
- core library imports
- environment checks
- project path configuration

In [2]:
# --- Imports ---

import pandas as pd
import sys
from src.paths import RAW_DIR, INTERIM_DIR, PROCESSED_DIR, ensure_data_dirs

In [3]:
# --- Environment check ---

print("Python version:", sys.version.split()[0])

try:
    import src
    print("Project package import: OK")
except ImportError:
    print("ERROR: project package 'src' not found")

try:
    import pandas as pd
    print("pandas version:", pd.__version__)
except ImportError:
    print("ERROR: pandas not installed")

Python version: 3.13.5
Project package import: OK
pandas version: 2.2.3


In [4]:
# --- Project paths configuration ---

ensure_data_dirs()

print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

RAW_DIR: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\1_raw
PROCESSED_DIR: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed


## 1.2 Data ingestion and validation

This section prepares the datasets used in the analysis.

Steps included:
- creation of the OECD member countries dataset
- loading of the raw GBD ASD prevalence dataset
- structural validation checks

In [5]:
# --- Create canonical OECD members dataset ---

oecd_members_list = ["Australia","Austria","Belgium","Canada","Chile","Colombia","Costa Rica",
                     "Czechia","Denmark","Estonia","Finland","France","Germany","Greece",
                     "Hungary","Iceland","Ireland","Israel","Italy","Japan","South Korea",
                     "Latvia","Lithuania","Luxembourg","Mexico","Netherlands","New Zealand",
                     "Norway","Poland","Portugal","Slovakia","Slovenia","Spain","Sweden",
                     "Switzerland","Turkey","United Kingdom","United States"]

oecd_members = pd.DataFrame({"country": oecd_members_list})

oecd_members.to_csv(PROCESSED_DIR / "oecd_members.csv", index=False)

print("Saved:", PROCESSED_DIR / "oecd_members.csv")

Saved: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed\oecd_members.csv


In [6]:
# --- Load and validate OECD members dataset ---

oecd_members = pd.read_csv(PROCESSED_DIR / "oecd_members.csv")

print("Rows:", len(oecd_members))
print("Columns:", list(oecd_members.columns))

oecd_members.head()

Rows: 38
Columns: ['country']


,country
0,Australia
1,Austria
2,Belgium
3,Canada
4,Chile


In [7]:
# --- Load and validate GBD autism prevalence dataset (OECD members selection) ---

gbd_file = RAW_DIR / "ihme_gbd_asd_prevalence_oecd_1990_2023.csv"

gbd_raw = pd.read_csv(gbd_file)

print("Rows:", len(gbd_raw))
print("Columns:", list(gbd_raw.columns))

Rows: 38760
Columns: ['population_group_id', 'population_group_name', 'measure_id', 'measure_name', 'location_id', 'location_name', 'sex_id', 'sex_name', 'age_id', 'age_name', 'cause_id', 'cause_name', 'metric_id', 'metric_name', 'year', 'val', 'upper', 'lower']


## 1.3 GBD download parameters

The dataset was downloaded manually from the IHME Global Burden of Disease (GBD) Results Tool.

Source:
https://vizhub.healthdata.org/gbd-results/

The OECD cohort used for the manual location selection is based on the official OECD members list:
https://www.oecd.org/about/members-and-partners/

Parameters used in the download:

Measure
- Prevalencia

Metric
- Tasa

Cause
- Trastornos del espectro autista

Sex
- Hombre
- Mujer

Age groups
- Menores de 5 años
- 5-9 años
- 10-14 años
- 15-19 años
- 20-24 años
- 25-29 años
- 30-34 años
- 35-39 años
- 40-44 años
- 45-49 años
- 50-54 años
- 55-59 años
- 60-64 años
- 65-69 años
- 70+ años

Years
- 1990-2023

Locations
- 38 OECD member countries, selected manually in the GBD interface

Downloaded file stored at:

`data/1_raw/ihme_gbd_asd_prevalence_oecd_1990_2023.csv`